# Extension: chemical representations (SMILES / RDKit)

Tests the paper's suggested future work — *"including chemical representation of
molecules into the model"* — by adding structure-derived features to the DILI-severity
models, with regularisation for the high-dimensional structure block.

- **Data**: same 184 compounds / 147-37 split / 3-class target.
- **Structure**: SMILES resolved from PubChem, encoded as RDKit **MACCS keys** (167 bits).
- **Blocks**: assay (8) / structure (167) / combined.
- **Models**: regularised logistic regression, random forest, gradient boosting;
  Bayesian POLR and BNN with a weak prior vs a **horseshoe** prior on the structure block.
- **Protocol**: fixed split, 20-bootstrap SDs.

The full run is in `scripts/structure_extension.py`; this notebook presents the results.

In [ ]:
import os
import pandas as pd

RESULTS = "../data/08_reporting/structure_extension_results.csv"
if not os.path.exists(RESULTS):
    import subprocess, sys
    subprocess.run([sys.executable, "../scripts/structure_extension.py"], check=True)
res = pd.read_csv(RESULTS)
print(res.shape)


## Ablation (frequentist, test set)

In [ ]:
freq = res[res["prior"] == "frequentist"][
    ["block", "model", "OBS", "BSS", "BA", "Acc", "BSS_sd", "BA_sd"]]
freq.round(3)


## Bayesian models (combined = 8 assay + 15 structure PCs)

In [ ]:
bayes = res[res["prior"] != "frequentist"][
    ["model", "prior", "WAIC", "OBS_test", "BSS_test", "BA_test", "Acc_test"]]
bayes.round(3)


## Findings

- **Structure carries signal** — structure-only models reach BA 0.69 / BSS 0.26, far
  above the frequency baseline.
- **Structure is complementary** — combined assay + structure is the best frequentist
  model (RF BA 0.695).
- **Regularisation is decisive** — the weak-prior combined BNN collapses (BA 0.456),
  but the horseshoe prior restores it to BA 0.605 with the best OOS OBS (0.145) and
  BSS (0.344) of any Bayesian model, improving on the assay-only BNN.
- **Caveat**: 37 test compounds — judge differences against the bootstrap SDs.